# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [10]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [8]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the provided information, exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch**: Start on your hands and knees. Alternate between arching your back up (cat position) and letting it sag down (cow position). Aim for 10-15 repetitions.\n\n2. **Bird Dog**: From hands and knees, extend opposite arm and leg simultaneously while keeping your core engaged. Hold each extension for about 5 seconds and switch sides. Do 10 repetitions per side.\n\n3. **Pelvic Tilts**: Lie on your back with knees bent. Flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\n4. **Partial Crunches**: Lie on your back with knees bent and arms crossed over your chest. Tighten your stomach muscles and lift your shoulders off the floor. Hold briefly, then lower back down. Do 8-12 repetitions.\n\n5. **Knee-to-Chest Stretch**: Lie on your back, pull one knee toward your chest while kee

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Proper sleep is crucial for physical repair, immune function, mental well-being, and cognitive processes like memory consolidation. During sleep, the body repairs tissues, releases hormones that regulate growth and appetite, and the brain processes and stores memories. Consistently getting 7-9 hours of quality sleep helps maintain a healthy immune system, supports emotional stability, and enhances learning and concentration. Conversely, poor sleep or sleep disturbances like insomnia can negatively affect physical health, mental health, and overall well-being.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils such as peppermint or lavender\n- Taking small amounts of caffeine (with caution, as it can help or hurt)\n- Maintaining a regular sleep schedule\n\nFor immediate stress relief, techniques like deep breathing, progressive muscle relaxation, grounding exercises, taking a short walk in nature, and listening to calming music can be helpful.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [20]:
!pip install rank_bm25

In [21]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on your hands and knees, then alternate arching your back up (like a cat) and letting it sag down (like a cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold the position for 5 seconds, then switch sides. Perform 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and may prevent future episodes.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health. It involves cycles of about 90 minutes, including REM and non-REM stages, with each stage serving specific functions. Deep sleep (Stage 3) is essential for body repair and regeneration. Adequate sleep (7-9 hours for adults) helps maintain immune function, supports mental health, and ensures proper nutrient absorption. Additionally, a good sleep environment—such as a comfortable mattress, appropriate temperature, darkness, and quietness—enhances sleep quality. Poor sleep or disorders like insomnia can negatively impact health, so establishing healthy sleep habits is important for overall wellness.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include relaxation techniques such as progressive muscle relaxation, meditation, and deep breathing exercises. Herbal teas like chamomile or valerian root may also help reduce stress and promote relaxation. Additionally, addressing triggers like dehydration and poor sleep through proper hydration and good sleep hygiene can help manage headaches caused by stress. However, it is always advisable to consult a healthcare professional before starting any new remedies.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

BM25 tends to outperform embedding search when the user query contains rare keywords, exact terms, IDs, codes, product names, or very specific phrases where lexical match is critical. For example:

> “What are the side effects of valerian root and how does it interact with SSRIs?”
or
> “What does 65–68°F refer to in sleep hygiene recommendations?”

In these cases, BM25 is strong because it directly rewards exact token overlap (“valerian root”, “SSRIs”, “65–68°F”). Embedding retrieval can sometimes return semantically related passages (e.g., general sleep tips) while missing the one chunk containing the exact term. BM25 is also typically better when the corpus has lots of near-duplicate content and you want the chunk with the exact phrase.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
import sys
!{sys.executable} -m pip install -U langchain-community langchain-text-splitters langchain-cohere

In [20]:
import sys
!{sys.executable} -m pip install -U cohere

In [21]:
import sys
!{sys.executable} -m pip install -U cohere

In [22]:
import os
import cohere

co = cohere.Client(os.environ["COHERE_API_KEY"])

def cohere_rerank(query: str, docs, top_n: int = 5, model: str = "rerank-v3.5"):
    texts = [d.page_content for d in docs]
    rr = co.rerank(model=model, query=query, documents=texts, top_n=top_n)
    return [docs[r.index] for r in rr.results]

def rerank_retriever(query: str, base_retriever, fetch_k: int = 20, top_n: int = 5):
    # If base retriever supports k as search_kwargs
    try:
        base_retriever.search_kwargs = {"k": fetch_k}
    except Exception:
        pass

    candidates = base_retriever.invoke(query)
    if not candidates:
        return []
    return cohere_rerank(query, candidates, top_n=top_n)


In [23]:
import sys
!{sys.executable} -m pip install -U truststore
import truststore
truststore.inject_into_ssl()

In [24]:
import os, cohere
co = cohere.Client(os.environ["COHERE_API_KEY"])


In [25]:
co.rerank(model="rerank-v3.5", query="test", documents=["a","b"], top_n=1)

RerankResponse(id='f26e9dda-868c-4ed3-bb82-bff614372531', results=[RerankResponseResultsItem(document=None, index=1, relevance_score=0.0784517)], meta=ApiMeta(api_version=ApiMetaApiVersion(version='1', is_deprecated=None, is_experimental=None), billed_units=ApiMetaBilledUnits(images=None, input_tokens=None, image_tokens=None, output_tokens=None, search_units=1.0, classifications=None), tokens=None, cached_tokens=None, warnings=None))

In [26]:
query = "How can I improve sleep quality?"
reranked_docs = rerank_retriever(query, naive_retriever, fetch_k=20, top_n=5)
reranked_docs

[Document(metadata={'source': 'data/HealthWellnessGuide.txt', '_id': 'e9e6410ae3a2447da7acaefc66c5b905', '_collection_name': 'wellness_guide'}, page_content='Chapter 8: Improving Sleep Quality\n\nSleep hygiene refers to habits and practices that promote consistent, quality sleep.\n\nEssential sleep hygiene practices:\n- Maintain a consistent sleep schedule, even on weekends\n- Create a relaxing bedtime routine (reading, gentle stretching, warm bath)\n- Keep your bedroom cool, dark, and quiet\n- Limit screen exposure 1-2 hours before bed\n- Avoid caffeine after 2 PM\n- Exercise regularly, but not too close to bedtime\n- Limit alcohol and heavy meals before bed'),
 Document(metadata={'source': 'data/HealthWellnessGuide.txt', '_id': '627d93209945449289f19787a357d3d8', '_collection_name': 'wellness_guide'}, page_content='Creating an optimal sleep environment:\n- Temperature: 65-68 degrees Fahrenheit (18-20 Celsius)\n- Darkness: Use blackout curtains or a sleep mask\n- Quiet: Consider white

Let's create our chain again, and see how this does!

In [27]:
!pip install langchain-community langchain-core langchain


In [28]:
class ManualCohereRerankRetriever:
    def __init__(self, base_retriever, fetch_k=20, top_n=5, model="rerank-v3.5"):
        self.base_retriever = base_retriever
        self.fetch_k = fetch_k
        self.top_n = top_n
        self.model = model

    def invoke(self, query: str):
        try:
            self.base_retriever.search_kwargs = {"k": self.fetch_k}
        except Exception:
            pass
        candidates = self.base_retriever.invoke(query)
        if not candidates:
            return []
        return cohere_rerank(query, candidates, top_n=self.top_n, model=self.model)

compression_retriever = ManualCohereRerankRetriever(naive_retriever, fetch_k=20, top_n=5)
docs = compression_retriever.invoke("How can I improve sleep quality?")
docs

[Document(metadata={'source': 'data/HealthWellnessGuide.txt', '_id': 'e9e6410ae3a2447da7acaefc66c5b905', '_collection_name': 'wellness_guide'}, page_content='Chapter 8: Improving Sleep Quality\n\nSleep hygiene refers to habits and practices that promote consistent, quality sleep.\n\nEssential sleep hygiene practices:\n- Maintain a consistent sleep schedule, even on weekends\n- Create a relaxing bedtime routine (reading, gentle stretching, warm bath)\n- Keep your bedroom cool, dark, and quiet\n- Limit screen exposure 1-2 hours before bed\n- Avoid caffeine after 2 PM\n- Exercise regularly, but not too close to bedtime\n- Limit alcohol and heavy meals before bed'),
 Document(metadata={'source': 'data/HealthWellnessGuide.txt', '_id': '627d93209945449289f19787a357d3d8', '_collection_name': 'wellness_guide'}, page_content='Creating an optimal sleep environment:\n- Temperature: 65-68 degrees Fahrenheit (18-20 Celsius)\n- Darkness: Use blackout curtains or a sleep mask\n- Quiet: Consider white

In [29]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer ONLY using the provided context. If the answer isn't in the context, say you don't know."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])

# 2) Create a chain (manual retrieval + LLM)

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

def retrieve_context(inputs: dict):
    # inputs is {"question": "..."}
    docs = compression_retriever.invoke(inputs["question"])
    return {
        "question": inputs["question"],
        "context": format_docs(docs),
        "docs": docs, # keep docs around if you want to inspect
    }

contextual_compression_retrieval_chain = (
    RunnableLambda(retrieve_context)
    | RunnableLambda(lambda x: {"question": x["question"], "context": x["context"], "docs": x["docs"]})
    | RunnableLambda(lambda x: {**x, "response": llm.invoke(prompt.format_messages(question=x["question"], context=x["context"]))})
)

In [30]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is crucial for physical health, mental well-being, and cognitive function. It allows the body to repair tissues, consolidate memories, and release hormones that regulate growth and appetite. Adults typically need 7-9 hours of sleep per night, and sleep occurs in cycles that include both REM and non-REM stages, each serving important functions for recovery and health.'

In [31]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include:\n\n- Deep breathing: Inhale for 4 counts, hold for 4, exhale for 4\n- Progressive muscle relaxation: Tense and release muscle groups from toes to head\n- Grounding technique: Name 5 things you see, 4 you hear, 3 you feel, 2 you smell, 1 you taste\n- Taking a short walk, preferably in nature\n- Listening to calming music\n\nFor headaches, natural remedies include:\n\n- Drinking water and staying hydrated\n- Applying a cold or warm compress to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of temples and neck\n- Using peppermint or lavender essential oils\n- Consuming caffeine in small amounts (can help or hurt)\n- Maintaining a regular sleep schedule'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [32]:
from typing import List
from langchain_core.documents import Document

def generate_multi_queries(question: str, llm, n: int = 4) -> List[str]:
    prompt = f"""Generate {n} diverse search queries that would retrieve relevant passages for answering the question.
Return ONLY the queries, one per line.

Question: {question}
"""
    out = llm.invoke(prompt)
    text = out.content if hasattr(out, "content") else str(out)
    queries = [q.strip("-• ").strip() for q in text.splitlines() if q.strip()]
    return queries[:n]

class ManualMultiQueryRetriever:
    def __init__(self, base_retriever, llm, n_queries: int = 4, k_per_query: int = 5):
        self.base_retriever = base_retriever
        self.llm = llm
        self.n_queries = n_queries
        self.k_per_query = k_per_query

    def invoke(self, query: str):
        # set k for each retrieval
        try:
            self.base_retriever.search_kwargs = {"k": self.k_per_query}
        except Exception:
            pass

        queries = generate_multi_queries(query, self.llm, n=self.n_queries)

        all_docs: List[Document] = []
        seen = set()

        for q in queries:
            docs = self.base_retriever.invoke(q)
            for d in docs:
                key = (d.page_content[:200], tuple(sorted((d.metadata or {}).items())))
                if key not in seen:
                    seen.add(key)
                    all_docs.append(d)

        return all_docs

multi_query_retriever = ManualMultiQueryRetriever(
    base_retriever=naive_retriever,
    llm=chat_model,
    n_queries=4,
    k_per_query=5
)

# test
multi_query_retriever.invoke("What exercises can help with lower back pain?")


[Document(metadata={'source': 'data/HealthWellnessGuide.txt', '_id': '342a505f92a848f7bd16ba5e096939d7', '_collection_name': 'wellness_guide'}, page_content='Recommended exercises for lower back pain include:\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.'),
 Document(metadata={'source': 'data/HealthWellnessGuide.txt', '_id': '575c2a4ebff64765aa35d109097aa176', '_collection_name': 'wellness_guide'}, page_content='Chapter 2: Exercises for Common Problems\n\nLower Back Pain Relief\nLower back pain affects approximately 80% of adults at some point in their lives. Gentle stretching and strengthening exercises can help alleviate discomfort and prevent future episodes.'),
 Document(metadata={'source': 'data/HealthWellnessGuide.txt', '_i

In [33]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Wrap your manual retriever in a Runnable
multi_query_runnable = RunnableLambda(lambda q: multi_query_retriever.invoke(q))

multi_query_retrieval_chain = (
    {
        "context": itemgetter("question") | multi_query_runnable,
        "question": itemgetter("question"),
    }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

multi_query_retrieval_chain.invoke(
    {"question": "What exercises can help with lower back pain?"}
)["response"].content

"Exercises that can help with lower back pain include:\n\n1. Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (like a cat) and letting it sag down (like a cow). Perform 10-15 repetitions.\n\n2. Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold this position for about 5 seconds, then switch sides. Aim for 10 repetitions per side.\n\n3. Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis upward. Hold for 10 seconds, and repeat 8-12 times.\n\nThese exercises are gentle stretches and strengthening movements designed to alleviate discomfort and help prevent future episodes of lower back pain. However, it's always best to consult with a healthcare professional before starting any new exercise routine, especially if you have existing health conditions."

In [34]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the provided information, here are some exercises that can help with lower back pain:\n\n1. Cat-Cow Stretch: Start on your hands and knees. Alternate between arching your back up (like a cat) and letting it sag down (like a cow). Do 10-15 repetitions.\n\n2. Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for 5 seconds, then switch sides. Aim for 10 repetitions per side.\n\n3. Pelvic Tilts: Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\n4. Partial Crunches: Lie on your back with knees bent, cross arms over your chest, tighten stomach muscles, and lift shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n5. Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat on the ground. Hold for 15-30 seconds and switch l

In [35]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (7-9 hours for adults) supports a strong immune system, helps manage stress, and promotes a balanced work-life relationship. Poor sleep or insomnia can lead to symptoms such as constant exhaustion, declining physical health, and negative effects on mental health. Practicing good sleep hygiene—like maintaining a consistent sleep schedule, creating a relaxing bedtime routine, and optimizing the sleep environment—can improve sleep quality and, consequently, overall health.'

In [36]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in progressive muscle relaxation, using grounding techniques (such as naming things you see, hear, feel, smell, and taste), taking short walks in nature, listening to calming music, staying hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, massaging the temples and neck, using essential oils like peppermint or lavender, maintaining a regular sleep schedule, and establishing calming evening routines like gentle stretching, warm baths, or reading.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Generating multiple reformulations of a user query improves recall because the same information can be expressed in many different ways in the source documents. A single query may miss relevant chunks if the wording in the document uses different synonyms, phrasing, or technical terms. By creating multiple versions of the same question, the retriever can search from different semantic angles and recover documents that would otherwise be missed.

This is especially helpful when there is a vocabulary mismatch between the user and the documents. For example, a user may ask about “lower back pain,” while a document may use terms like “lumbar discomfort” or “back strain.” Multiple reformulations increase the chance of matching those variations. As a result, recall improves because more relevant documents are retrieved, although this can sometimes increase cost and slightly reduce precision if too many loosely related results are pulled in.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [37]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
import uuid

# Splitters
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

# Qdrant in-memory (child chunks only)
client = QdrantClient(location=":memory:")
client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE),
)

child_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child",
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    client=client,
)

# Parent doc store: parent_id -> parent Document
parent_store = {}

# raw_docs must be List[Document]
# If your notebook uses docs instead of raw_docs, swap variable name here.
parents = []
for d in raw_docs:
    # Treat each raw doc as a parent (you can also parent-split here if needed)
    pid = str(uuid.uuid4())
    parent_doc = Document(page_content=d.page_content, metadata={**(d.metadata or {}), "parent_id": pid})
    parent_store[pid] = parent_doc
    parents.append(parent_doc)

# Create child chunks with parent_id metadata
child_docs = []
for p in parents:
    chunks = child_splitter.split_text(p.page_content)
    for c in chunks:
        child_docs.append(Document(page_content=c, metadata={"parent_id": p.metadata["parent_id"]}))

# Index child chunks
child_vectorstore.add_documents(child_docs)

print(f"Indexed parents: {len(parent_store)} | child chunks: {len(child_docs)}")


Indexed parents: 1 | child chunks: 56


We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [38]:
from typing import List, Dict, Any

class ManualParentDocumentRetriever:
    def __init__(self, child_vectorstore: QdrantVectorStore, parent_store: Dict[str, Document], k_children: int = 12, k_parents: int = 5):
        self.child_vectorstore = child_vectorstore
        self.parent_store = parent_store
        self.k_children = k_children
        self.k_parents = k_parents

    def invoke(self, query: str) -> List[Document]:
        # retrieve child chunks
        child_hits = self.child_vectorstore.similarity_search(query, k=self.k_children)
        if not child_hits:
            return []

        # collect parent ids in rank order
        parent_ids = []
        seen = set()
        for ch in child_hits:
            pid = (ch.metadata or {}).get("parent_id")
            if pid and pid not in seen:
                seen.add(pid)
                parent_ids.append(pid)

        # return parent docs
        parents = [self.parent_store[pid] for pid in parent_ids if pid in self.parent_store]
        return parents[: self.k_parents]


parent_document_retriever = ManualParentDocumentRetriever(
    child_vectorstore=child_vectorstore,
    parent_store=parent_store,
    k_children=15,
    k_parents=5,
)

In [39]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Wrap the manual retriever into a Runnable so `|` works
parent_retriever_runnable = RunnableLambda(lambda q: parent_document_retriever.invoke(q))

parent_document_retrieval_chain = (
    {
        "context": itemgetter("question") | parent_retriever_runnable,
        "question": itemgetter("question"),
    }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

parent_document_retrieval_chain.invoke(
    {"question": "What exercises can help with lower back pain?"}
)["response"].content

'Exercises that can help with lower back pain include gentle stretching and strengthening movements such as:\n\n- **Cat-Cow Stretch:** On hands and knees, alternate arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten your stomach muscles, and lift your shoulders off the floor. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, and repeat 8-12 times.\n\nThese exercises are recommended for relieving lower back disc

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [40]:
from typing import Any, Iterable, List, Tuple
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
import uuid
from typing import Any, Iterable, List, Tuple

class SimpleInMemoryStore:
    def __init__(self):
        self._db = {}

    def mset(self, items: Iterable[Tuple[str, Any]]):
        for k, v in items:
            self._db[k] = v

    def mget(self, keys: List[str]):
        return [self._db.get(k) for k in keys]

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

client = QdrantClient(location=":memory:")
client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE),
)

child_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child",
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    client=client,
)

store = SimpleInMemoryStore() # ✅ docstore

# raw_docs must be List[Document]
parent_docs = []
for d in raw_docs:
    pid = str(uuid.uuid4())
    parent_doc = Document(
        page_content=d.page_content,
        metadata={**(d.metadata or {}), "parent_id": pid},
    )
    parent_docs.append(parent_doc)

store.mset([(p.metadata["parent_id"], p) for p in parent_docs])

child_docs = []
for p in parent_docs:
    for c in child_splitter.split_text(p.page_content):
        child_docs.append(Document(page_content=c, metadata={"parent_id": p.metadata["parent_id"]}))

child_vectorstore.add_documents(child_docs)

print(f"Parents stored: {len(parent_docs)} | Child chunks indexed: {len(child_docs)}")

class ManualParentDocumentRetrieverWithStore:
    def __init__(self, vectorstore, docstore, k_children=15, k_parents=5):
        self.vectorstore = vectorstore
        self.docstore = docstore
        self.k_children = k_children
        self.k_parents = k_parents

    def invoke(self, query: str):
        child_hits = self.vectorstore.similarity_search(query, k=self.k_children)
        if not child_hits:
            return []

        parent_ids = []
        seen = set()
        for ch in child_hits:
            pid = (ch.metadata or {}).get("parent_id")
            if pid and pid not in seen:
                seen.add(pid)
                parent_ids.append(pid)

        parents = self.docstore.mget(parent_ids)
        parents = [p for p in parents if p is not None]
        return parents[: self.k_parents]

parent_document_retriever = ManualParentDocumentRetrieverWithStore(
    vectorstore=child_vectorstore,
    docstore=store,
    k_children=15,
    k_parents=5,
)

from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

parent_retriever_runnable = RunnableLambda(lambda q: parent_document_retriever.invoke(q))

parent_document_retrieval_chain = (
    {
        "context": itemgetter("question") | parent_retriever_runnable,
        "question": itemgetter("question"),
    }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

parent_document_retrieval_chain.invoke(
    {"question": "What exercises can help with lower back pain?"}
)["response"].content


Parents stored: 1 | Child chunks indexed: 56


'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate arching your back up (like a cat) and letting it sag down (like a cow). Do 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while engaging your core. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten your stomach muscles, and lift shoulders off the floor. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises are recommended to alleviate lower back discomfort and

By default, this is empty as we haven't added any documents - let's add some now!

In [41]:
new_raw_docs = raw_docs[:2] # pick any subset

In [42]:
# Add new parent docs
new_parent_docs = []
for d in new_raw_docs: # List[Document]
    pid = str(uuid.uuid4())
    pdoc = Document(page_content=d.page_content, metadata={**(d.metadata or {}), "parent_id": pid})
    new_parent_docs.append(pdoc)

store.mset([(p.metadata["parent_id"], p) for p in new_parent_docs])

# Add their child chunks
new_child_docs = []
for p in new_parent_docs:
    for c in child_splitter.split_text(p.page_content):
        new_child_docs.append(Document(page_content=c, metadata={"parent_id": p.metadata["parent_id"]}))

child_vectorstore.add_documents(new_child_docs)

['aed05eeac9a94fbd9aa32849668889f0',
 '8dae8fbe8a5a4b0c82018d747124df09',
 'a75944c16d3a487785e4662ab5297894',
 '14392c358bf64105a21153f6b3b442f9',
 '5d036abe043f413aa911642bb5e0f1ab',
 '32e5ba02a27a45478752834e01ecff5f',
 'b33aa5c9fcb14f758a30afd4dec1c8e4',
 'c3ea72d5899b437baf601aa5f4070c2c',
 'ae0dae65c0454b148144a095e174b3f8',
 '7bd92455712c4263a348407af7df9654',
 'f4ede11ceb97497683a732d21ef1cc87',
 '5697d63628c64aabbd33885bcfe56a02',
 'ec64f05c27ea479c9e397016422937f3',
 'b31fef72b76546b582d1deba3fb2b3e3',
 'a93e65dba75346698c3902c2425e9fde',
 '02ca9f099b734842911a0b015d7984ff',
 '1650402b7a2a4158971fca44a53cdf8d',
 '609a1b3f2c6547739bf4d943f901d31b',
 '3ae0860453dc4d59844603bccc77d04f',
 'be1b6e19e0bc47edb449ad617c10a1e5',
 '480045ae286b40469e9c82925bdd06c3',
 '8eeea2ddbe29473db4442ddaa085d9c7',
 '13369f0386524174b798de017771eee8',
 '0ce646b1cbaf43219bb09eec86535fd6',
 'f3a27f5c3c0a4d70a441c6e4968a4fa8',
 'f35ea868b73545f5bac4262676a67a9a',
 'abd5b9f0ff30464fa2fcccd045abe826',
 

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [43]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

parent_document_retriever_runnable = RunnableLambda(lambda q: parent_document_retriever.invoke(q))

parent_document_retrieval_chain = (
    {
        "context": itemgetter("question") | parent_document_retriever_runnable,
        "question": itemgetter("question"),
    }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

parent_document_retrieval_chain.invoke(
    {"question": "What exercises can help with lower back pain?"}
)["response"].content


"The exercises that can help with lower back pain, as recommended in the guide, include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.  \n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.  \n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.  \n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.  \n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.  \n\nThese gentle stretching and strengthening exercises can

Let's give it a whirl!

In [44]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate discomfort and prevent future episodes of lower 

In [45]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

"Sleep has a significant impact on overall health. During sleep, your body repairs tissues, consolidates memories, and releases hormones that help regulate growth and appetite. Adequate sleep—typically 7-9 hours per night for adults—supports physical health, mental well-being, and cognitive functions. Good sleep quality, maintained through proper sleep hygiene practices such as a consistent sleep schedule, creating a relaxing bedtime routine, and ensuring a sleep-friendly environment, enhances your body's ability to recover, reduces the risk of chronic diseases, and improves mood and mental clarity. Conversely, poor or insufficient sleep can lead to problems such as fatigue, impaired immune function, and increased susceptibility to illnesses."

In [46]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises: Inhale for 4 counts, hold for 4, exhale for 4 to help calm the nervous system.\n- Progressive muscle relaxation: Tense and then relax muscle groups from toes to head to reduce physical tension.\n- Grounding techniques: Name 5 things you see, 4 you hear, 3 you feel, 2 you smell, and 1 you taste to anchor yourself in the present moment.\n- Rest in a dark, quiet room to alleviate headache symptoms.\n- Apply cold or warm compresses to the head, neck, or temples.\n- Gentle massage of the temples and neck can help relieve tension.\n- Use essential oils like peppermint or lavender, which are known for their calming effects.\n- Stay hydrated by drinking plenty of water.\n- Maintain regular sleep schedules and practice good sleep hygiene.\n- Herbal teas such as chamomile or valerian root may promote relaxation.\n- For headaches related to stress, managing triggers like dehydration, poor sleep, or eye strain 

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [47]:
from typing import List
from langchain_core.documents import Document

def _invoke_any(retriever, query: str) -> List[Document]:
    """Works for LangChain retrievers + your manual retrievers."""
    # Prefer .invoke if present
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    # Otherwise try callable
    if callable(retriever):
        return retriever(query)
    raise TypeError(f"Retriever {type(retriever)} has no invoke() and is not callable")

class ManualEnsembleRetriever:
    def __init__(self, retrievers: List, weights: List[float] | None = None, k: int = 10):
        self.retrievers = retrievers
        self.weights = weights or [1.0 / len(retrievers)] * len(retrievers)
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        scored = [] # (score, doc)
        for r, w in zip(self.retrievers, self.weights):
            docs = _invoke_any(r, query)
            # simple scoring: earlier docs from a retriever get slightly higher score
            for rank, d in enumerate(docs):
                score = w * (1.0 / (rank + 1))
                scored.append((score, d))

        # dedupe by content + metadata
        seen = set()
        merged = []
        for score, d in sorted(scored, key=lambda x: x[0], reverse=True):
            key = (d.page_content[:300], tuple(sorted((d.metadata or {}).items())))
            if key not in seen:
                seen.add(key)
                merged.append(d)
            if len(merged) >= self.k:
                break
        return merged


retriever_list = [
    bm25_retriever,
    naive_retriever,
    parent_document_retriever, # your manual parent retriever
    compression_retriever, # your manual reranker
    multi_query_retriever # your manual multiquery retriever
]

equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = ManualEnsembleRetriever(
    retrievers=retriever_list,
    weights=equal_weighting,
    k=10
)

# test
docs = ensemble_retriever.invoke("What exercises can help with lower back pain?")
docs[:2]

[Document(metadata={'source': 'data/HealthWellnessGuide.txt'}, page_content='Recommended exercises for lower back pain include:\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.'),
 Document(metadata={'source': 'data/HealthWellnessGuide.txt', '_id': '342a505f92a848f7bd16ba5e096939d7', '_collection_name': 'wellness_guide'}, page_content='Recommended exercises for lower back pain include:\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.')]

We'll pack *all* of these retrievers together in an ensemble.

In [49]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

ensemble_retriever_runnable = RunnableLambda(lambda q: ensemble_retriever.invoke(q))

ensemble_retrieval_chain = (
    {
        "question": itemgetter("question"),
        "context": itemgetter("question") | ensemble_retriever_runnable,
    }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {
        "response": rag_prompt | chat_model,
        "context": itemgetter("context"),
    }
)

# test
out = ensemble_retrieval_chain.invoke({"question": "What exercises can help with lower back pain?"})
out["response"]

AIMessage(content="Exercises that can help with lower back pain include:\n\n1. Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n2. Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n4. Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n5. Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, and repeat 8-12 times.\n\nThese exercises, performed gently and regularly, can help alleviate and p

Let's look at our results!

In [50]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. Cat-Cow Stretch:\n   - Start on your hands and knees.\n   - Alternate between arching your back up (like a cat) and letting it sag down (like a cow).\n   - Do 10-15 repetitions.\n\n2. Bird Dog:\n   - From hands and knees position.\n   - Extend opposite arm and leg while keeping your core engaged.\n   - Hold each extension for about 5 seconds.\n   - Switch sides.\n   - Perform 10 repetitions per side.\n\n3. Partial Crunches:\n   - Lie on your back with knees bent.\n   - Cross arms over your chest.\n   - Tighten your stomach muscles and lift your shoulders off the floor.\n   - Hold briefly, then gently lower back down.\n   - Do 8-12 repetitions.\n\n4. Knee-to-Chest Stretch:\n   - Lie on your back.\n   - Pull one knee toward your chest, keeping the other foot flat on the floor.\n   - Hold for 15-30 seconds.\n   - Switch legs.\n\n5. Pelvic Tilts:\n   - Lie on your back with knees bent.\n   - Flatten your back against the floor by

In [51]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Adequate and quality sleep (generally 7-9 hours per night) is essential for various bodily functions, including tissue repair, memory consolidation, hormone regulation, and immune system support. During sleep, the body cycles through different stages, such as light sleep, deep sleep, and REM sleep, each playing a vital role in physical and mental wellness. Poor sleep or sleep disturbances like insomnia can lead to health issues such as weakened immunity, cognitive impairments, mood problems, and increased risk of chronic diseases. Practicing good sleep hygiene—like maintaining a consistent schedule, creating a restful environment, and avoiding screens before bed—can significantly improve sleep quality and overall health.'

In [52]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (e.g., inhaling for 4 counts, holding for 4, exhaling for 4)\n- Progressive muscle relaxation, tensing and releasing muscle groups\n- Grounding techniques, such as identifying 5 things you see, 4 you hear, 3 you feel, 2 you smell, and 1 you taste\n- Taking short walks, preferably in nature\n- Listening to calming music\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of temples and neck\n- Using essential oils like peppermint or lavender\n\nStaying well-hydrated by drinking water, maintaining a regular sleep schedule, and practicing mindfulness and relaxation techniques can also help manage headaches and stress naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [53]:
!pip install -U langchain-experimental

In [54]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [55]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [56]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [57]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [58]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [59]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises are recommended for alleviating lower back discomfort and preventing future episodes.'

In [60]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults typically need 7-9 hours of sleep per night. Good sleep quality is achieved through proper sleep hygiene practices, such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine, and optimizing the sleep environment (cool, dark, quiet). Poor sleep can lead to symptoms like fatigue, headaches, and impaired cognitive function, while adequate, restful sleep helps improve mood, immune function, and overall physical health.'

In [61]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking plenty of water to stay hydrated.\n- Applying cold or warm compresses to the head or neck.\n- Resting in a dark, quiet room.\n- Gentle massage of the temples and neck.\n- Using essential oils such as peppermint or lavender.\n- Practicing relaxation techniques like deep breathing exercises and progressive muscle relaxation.\n- Engaging in short walks, especially in nature.\n- Listening to calming music.\n- Managing stress through regular exercise, social support, mindfulness, and hobbies.\n\nThese approaches can help alleviate symptoms naturally and promote overall well-being.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

If sentences are short and highly repetitive, semantic chunking may behave poorly because many adjacent sentences will look very similar in embedding space. This can cause the algorithm to merge multiple FAQ items into one large chunk, even when they actually belong to different questions. In other cases, the embeddings may not contain enough signal to identify clean topic boundaries, so chunking becomes unstable or less meaningful.

To adjust for this, I would make the chunking more conservative and rely more on document structure than pure semantic similarity. For example, I would chunk by each FAQ question-answer pair, headings, bullet boundaries, or separators instead of only using embedding similarity. I would also reduce the maximum chunk size, require stronger similarity before merging sentences, and use less overlap. In highly repetitive FAQ-style data, a hybrid or rule-based chunking strategy is usually more reliable than pure semantic chunking.


---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [3]:
!pip install -U ragas langchain langchain-community datasets langchain-openai openai

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.1 MB 8.1 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 3.8 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.24.0
    Uninstalling openai-2.24.0:
      Successfully uninstalled openai-2.24.0


In [11]:
from collections import defaultdict
from langchain_core.documents import Document

def merge_by_source(docs):
    by_src = defaultdict(list)

    for d in docs:
        src = (d.metadata or {}).get("source", "unknown")
        by_src[src].append(d.page_content)

    merged = []
    for src, parts in by_src.items():
        text = "\n\n".join(parts)
        merged.append(Document(page_content=text, metadata={"source": src}))

    return merged

# build long docs from your existing chunked docs
docs_long = merge_by_source(raw_docs)

# keep only reasonably long docs
docs_long = [d for d in docs_long if len(d.page_content) >= 1200]

print("Number of long docs:", len(docs_long))
if docs_long:
    print("Example doc length:", len(docs_long[0].page_content))
    print(docs_long[0].metadata)
else:
    print("docs_long is empty")

Number of long docs: 1
Example doc length: 16206
{'source': 'data/HealthWellnessGuide.txt'}


In [ ]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.testset import TestsetGenerator

# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "YOUR_KEY")

llm = ChatOpenAI(model="gpt-4o-mini")

# ✅ LangChain embeddings (supports async methods expected by Ragas)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

generator = TestsetGenerator.from_langchain(
    llm=llm,
    embedding_model=embedding_model,
)

TESTSET_SIZE = 60
testset = generator.generate_with_langchain_docs(docs_long, testset_size=TESTSET_SIZE)
df_golden = testset.to_pandas()
df_golden.head()


Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 1216.80it/s]
Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.
Generating Samples: 100%|██████████| 54/54 [01:02<00:00,  1.15s/it]


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,How to relieve lower bak pain?,[The Personal Wellness Guide\nA Comprehensive ...,"To relieve lower back pain, gentle stretching ...",Health and Wellness Coach,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
1,Wat is a Personal Wellness Gude?,[The Personal Wellness Guide\nA Comprehensive ...,A Personal Wellness Guide is a comprehensive r...,Health and Wellness Coach,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
2,What are some exercises that can help with nek...,[The Personal Wellness Guide\nA Comprehensive ...,To relieve neck and shoulder tension caused by...,Health and Wellness Coach,MISSPELLED,LONG,single_hop_specific_query_synthesizer
3,How does exercise contribute to overall well-b...,[The Personal Wellness Guide\nA Comprehensive ...,Exercise is one of the most important things y...,Health and Wellness Coach,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
4,How exercise help with well-being and what typ...,[The Personal Wellness Guide\nA Comprehensive ...,Exercise is one of the most important things y...,Health and Wellness Coach,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer


In [13]:
from datasets import Dataset
import pandas as pd
import numpy as np
import time
from operator import itemgetter

def detect_reference_column(df: pd.DataFrame):
    candidates = ["ground_truth", "reference", "answer", "expected_answer", "response"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find reference column. Available columns: {list(df.columns)}")

ref_col = detect_reference_column(df_golden)

gold_questions = df_golden["user_input"].tolist()
gold_truth = df_golden[ref_col].tolist()

print("Using reference column:", ref_col)
print("Golden size:", len(df_golden))

Using reference column: reference
Golden size: 54


In [14]:
from langchain_community.callbacks.manager import get_openai_callback

def safe_invoke(retriever, query: str):
    # works with .invoke or callable
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if callable(retriever):
        return retriever(query)
    raise TypeError(f"Retriever {type(retriever)} not invokable")

def run_retriever(df_golden, retriever, name: str, k: int = 10):
    rows = []
    latencies = []
    token_rows = []
    
    for q in df_golden["user_input"]:
        # cost tracking (OpenAI only) - if retriever doesn't call OpenAI, tokens = 0
        try:
            with get_openai_callback() as cb:
                t0 = time.perf_counter()
                docs = safe_invoke(retriever, q)
                dt = time.perf_counter() - t0
            tokens = cb.total_tokens
            usd_cost = cb.total_cost
        except Exception:
            # fallback: no callback available / non-openai / etc.
            t0 = time.perf_counter()
            docs = safe_invoke(retriever, q)
            dt = time.perf_counter() - t0
            tokens = 0
            usd_cost = 0.0

        docs = docs[:k]
        ctxs = [d.page_content for d in docs]
        
        rows.append({"user_input": q, "contexts": ctxs})
        latencies.append(dt)
        token_rows.append({"total_tokens": tokens, "usd_cost_est": usd_cost})

    return rows, latencies, token_rows

In [28]:
!pip install -U langchain langchain-core langchain-community langchain-classic langchain-openai langchain-text-splitters rank_bm25 langchain-cohere cohere

In [40]:
from langchain_community.retrievers import BM25Retriever

from langchain_classic.retrievers.parent_document_retriever import ParentDocumentRetriever
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

from langchain_cohere import CohereRerank

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# And here is the clean mapping for your dictionary:

# 1) BM25
bm25_retriever = BM25Retriever.from_documents(raw_docs)

# IMPORTANT:
# For a fair comparison, try to use the SAME corpus granularity for all retrievers.
# If your vectorstore was built on chunked docs like split_docs/chunks, use that for BM25 too.


# 2) Dense similarity retriever
# Assumes you already have a vectorstore built on the same docs/chunks
# Use child_vectorstore which was already created earlier
naive_retriever = child_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# 3) Dense MMR retriever
mmr_retriever = child_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5}
)

# 4) Optional hybrid ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, mmr_retriever],
    weights=[0.5, 0.5]
)

retrievers = {
    "bm25": bm25_retriever,
    "similarity": naive_retriever,
    "mmr": mmr_retriever,
    "ensemble": ensemble_retriever,
}

In [42]:
import pandas as pd

def detect_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

q_col = detect_col(df_golden, ["question", "query", "user_input", "prompt", "input"])
ref_col = detect_col(df_golden, ["ground_truth", "reference", "answer", "expected_answer", "response", "reference_answer"])

if q_col is None or ref_col is None:
    raise ValueError(f"Could not detect question/reference columns. Found: {df_golden.columns.tolist()}")

df_eval = df_golden.rename(columns={q_col: "question", ref_col: "ground_truth"}).copy()

print("Question column:", q_col)
print("Reference column:", ref_col)
print("Rows:", len(df_eval))

df_eval[["question", "ground_truth"]].head()


Question column: user_input
Reference column: reference
Rows: 54


,question,ground_truth
0,How to relieve lower bak pain?,"To relieve lower back pain, gentle stretching ..."
1,Wat is a Personal Wellness Gude?,A Personal Wellness Guide is a comprehensive r...
2,What are some exercises that can help with nek...,To relieve neck and shoulder tension caused by...
3,How does exercise contribute to overall well-b...,Exercise is one of the most important things y...
4,How exercise help with well-being and what typ...,Exercise is one of the most important things y...


In [43]:
import time
import numpy as np
from datasets import Dataset
from ragas import evaluate

# Ragas metric import that works across versions
try:
    from ragas.metrics import context_precision, context_recall, context_entity_recall
    ragas_metrics = [context_precision, context_recall, context_entity_recall]
except Exception:
    from ragas.metrics.collections import ContextPrecision, ContextRecall, ContextEntityRecall
    ragas_metrics = [ContextPrecision(), ContextRecall(), ContextEntityRecall()]

def invoke_any(retriever, query: str):
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if callable(retriever):
        return retriever(query)
    raise TypeError(f"{type(retriever)} is not invokable")

def evaluate_one_retriever(name, retriever, df_eval, k=5):
    rows = []
    latencies = []

    for question, reference in zip(df_eval["question"], df_eval["ground_truth"]):
        t0 = time.perf_counter()
        docs = invoke_any(retriever, question)[:k]
        dt = time.perf_counter() - t0

        retrieved_texts = [d.page_content for d in docs]

        # include both old and new column names for Ragas compatibility
        rows.append({
            "question": question,
            "user_input": question,
            "ground_truth": reference,
            "reference": reference,
            "contexts": retrieved_texts,
            "retrieved_contexts": retrieved_texts,
        })
        latencies.append(dt)

    eval_ds = Dataset.from_list(rows)

    result = evaluate(
        dataset=eval_ds,
        metrics=ragas_metrics,
        llm=llm,
        embeddings=embedding_model,
        raise_exceptions=False,
    )

    return result.to_pandas(), latencies

ragas_results = {}
latency_map = {}

for name, retriever in retrievers.items():
    print(f"Evaluating {name} ...")
    res_df, latencies = evaluate_one_retriever(name, retriever, df_eval, k=5)
    ragas_results[name] = res_df
    latency_map[name] = {
        "avg_latency_s": float(np.mean(latencies)),
        "p95_latency_s": float(np.percentile(latencies, 95)),
    }

print("Done.")

C:\Users\SinhaK\AppData\Local\Temp\ipykernel_54196\3014681517.py:8: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, context_entity_recall
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_54196\3014681517.py:8: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, context_entity_recall
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_54196\3014681517.py:8: DeprecationWarning: Importing context_entity_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from raga

Evaluating bm25 ...


Evaluating: 100%|██████████| 162/162 [04:45<00:00,  1.76s/it]


Evaluating similarity ...


Evaluating: 100%|██████████| 162/162 [10:38<00:00,  3.94s/it]


Evaluating mmr ...


Evaluating: 100%|██████████| 162/162 [09:00<00:00,  3.33s/it]


Evaluating ensemble ...


Evaluating: 100%|██████████| 162/162 [09:40<00:00,  3.58s/it]


Done.


In [44]:
import pandas as pd
import numpy as np

def pick_metric(d, candidates):
    for c in candidates:
        if c in d:
            return d[c]
    return np.nan

summary_rows = []

for name, res_df in ragas_results.items():
    m = res_df.mean(numeric_only=True).to_dict()

    context_precision_val = pick_metric(m, ["context_precision", "ContextPrecision"])
    context_recall_val = pick_metric(m, ["context_recall", "ContextRecall"])
    context_entity_recall_val = pick_metric(m, ["context_entity_recall", "ContextEntityRecall"])

    # For these four retrievers, query-time LLM calls are effectively zero.
    # So cost is in the same "low" bucket; latency and retrieval quality matter more.
    summary_rows.append({
        "retriever": name,
        "context_precision": context_precision_val,
        "context_recall": context_recall_val,
        "context_entity_recall": context_entity_recall_val,
        "avg_latency_s": latency_map[name]["avg_latency_s"],
        "p95_latency_s": latency_map[name]["p95_latency_s"],
        "cost_bucket": "low",
    })

df_summary = pd.DataFrame(summary_rows)

# performance-heavy ranking, then penalize slower retrievers
df_summary["perf_score"] = (
    0.50 * df_summary["context_precision"] +
    0.35 * df_summary["context_recall"] +
    0.15 * df_summary["context_entity_recall"]
)

df_summary["latency_penalty"] = df_summary["p95_latency_s"] / df_summary["p95_latency_s"].min()
df_summary["overall_score"] = df_summary["perf_score"] / df_summary["latency_penalty"]

df_ranked = df_summary.sort_values("overall_score", ascending=False).reset_index(drop=True)
df_ranked

,retriever,context_precision,context_recall,context_entity_recall,avg_latency_s,p95_latency_s,cost_bucket,perf_score,latency_penalty,overall_score
0,bm25,1.000000,0.972222,0.066215,0.000488,0.000866,low,0.850210,1.000000,0.850210
1,mmr,0.832922,0.754850,0.418919,0.438231,0.616344,low,0.743496,711.495352,0.001045
2,ensemble,0.964815,0.975309,0.063051,0.539269,1.016343,low,0.833223,1173.247689,0.000710
3,similarity,0.876235,0.913580,0.509870,0.287928,1.026573,low,0.834351,1185.056818,0.000704


In [45]:
best = df_ranked.iloc[0]

reason_map = {
    "bm25": "This suggests the corpus is relatively keyword-heavy, so lexical matching works very well.",
    "similarity": "This suggests semantic matching matters more than exact word overlap for this corpus.",
    "mmr": "This suggests reducing near-duplicate chunks helps, so diversity in retrieved results matters.",
    "ensemble": "This suggests combining lexical and dense retrieval gives the best balance for this corpus.",
}

paragraph = f"""
For this dataset, {best['retriever']} performed best overall. It achieved the strongest balance of retrieval quality and speed,
with Context Precision = {best['context_precision']:.3f}, Context Recall = {best['context_recall']:.3f}, and
Context Entity Recall = {best['context_entity_recall']:.3f}. Its p95 latency was {best['p95_latency_s']:.3f} seconds.
Among the retrievers evaluated here, query-time cost is effectively in the same low-cost bucket because BM25, similarity,
MMR, and ensemble retrieval do not add extra LLM calls during retrieval. So for this comparison, the main tradeoff is
performance versus latency rather than API spend. {reason_map.get(best['retriever'], '')}
""".strip()

print(paragraph)

For this dataset, bm25 performed best overall. It achieved the strongest balance of retrieval quality and speed,
with Context Precision = 1.000, Context Recall = 0.972, and
Context Entity Recall = 0.066. Its p95 latency was 0.001 seconds.
Among the retrievers evaluated here, query-time cost is effectively in the same low-cost bucket because BM25, similarity,
MMR, and ensemble retrieval do not add extra LLM calls during retrieval. So for this comparison, the main tradeoff is
performance versus latency rather than API spend. This suggests the corpus is relatively keyword-heavy, so lexical matching works very well.
